# HydroSight TN - Validation

This notebook prepares a holdout split, evaluates the classifier, and exports the final overlay-ready products.

In [ ]:
from pathlib import Path
import sys
import pandas as pd
from sklearn.model_selection import train_test_split
import joblib

project_root = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

from src import CLASS_COLORS, FEATURE_ORDER
from src.validate import validate_model
from src.visualize import plot_area_statistics, raster_to_rgb_png

output_dir = project_root / 'outputs'
model_path = project_root / 'models' / 'gwpz_xgboost.pkl'
training_csv = output_dir / 'training_data.csv'

df = pd.read_csv(training_csv)
train_df, test_df = train_test_split(df, test_size=0.2, random_state=42, stratify=df['label'])
test_csv = output_dir / 'test_data.csv'
test_df.to_csv(test_csv, index=False)
model = joblib.load(model_path)
test_df['ml_score'] = model.predict_proba(test_df[FEATURE_ORDER])[:, 1]
test_df.to_csv(test_csv, index=False)
metrics = validate_model(model, test_csv, output_dir=output_dir)
metrics

In [ ]:
raster_to_rgb_png(output_dir / 'gwpz_xgboost.tif', CLASS_COLORS, output_dir / 'gwpz_xgboost_rgb.png')
plot_area_statistics(output_dir / 'gwpz_xgboost.tif', output_dir / 'gwpz_statistics_ml.png')